In [1]:
import warnings
from pathlib import Path

import numpy as np
import prism

from imagematerials import buildings as bld
from imagematerials import vehicles as vhc
from imagematerials.concepts import create_building_graph, create_vehicle_graph
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import GenericMaterials, GenericStocks, MaterialIntensities
from imagematerials.util import (
    export_to_netcdf,
    import_from_netcdf,
    read_circular_economy_config,
    read_climate_policy_config,
    rebroadcast_prep_data,
)
# from imagematerials.vehicles import (
    # preprocess,
# )


In [2]:
def get_vema_prep():
    base_dir = Path("..", "data", "raw")
    prep_fp = Path("prep_vema.nc")
    if not prep_fp.is_file():
        climate_policy_scenario_dir = base_dir / 'SSP2'
        circular_economy_scenario_dirs = {"slow": base_dir / 'circular_economy_scenarios' / 'slow'}

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            climate_policy_config = read_climate_policy_config(climate_policy_scenario_dir)
            circular_economy_config = read_circular_economy_config(circular_economy_scenario_dirs)
            prep_data = vhc.preprocess(base_dir, climate_policy_config, circular_economy_config)

        export_to_netcdf(prep_data, prep_fp)

    prep_data = import_from_netcdf(prep_fp)
    share_coords = set()
    for cur_type in prep_data["shares"].Type.values:
        share_coords.add(cur_type.split(" - ")[0])
    output_coords_type = [x for x in prep_data["stocks"].Type.values if x not in share_coords] + list(prep_data["shares"].coords["Type"].values)
    prep_data.pop("shares")
    knowledge_graph = create_vehicle_graph()
    new_prep_data = rebroadcast_prep_data(prep_data, knowledge_graph, dim="Type", output_coords=output_coords_type)
    region_coords = np.sort(prep_data["stocks"].coords["Region"].values.astype(int)).astype(str)[:-2]
    new_prep_data = rebroadcast_prep_data(new_prep_data, knowledge_graph, dim="Region", output_coords=region_coords)
    new_prep_data["knowledge_graph"] = knowledge_graph

    new_prep_data["weights"] = new_prep_data.pop("vehicle_weights")
    new_prep_data["shares"] = None
    sec_vhc = Sector("vhc", new_prep_data)
    return sec_vhc

In [3]:
def get_buildings_prep():
    base_dir = Path("..", "IMAGE-Mat_old_version", "IMAGE-Mat", "BUMA")
    prep_fp = Path("prep_buildings.nc")
    if not prep_fp.is_file():
        prep_data = bld.preprocess(base_dir)
        export_to_netcdf(prep_data, prep_fp)
    else:
        prep_data = import_from_netcdf(prep_fp)
    new_prep_data = {k: v for k, v in prep_data.items()}
    new_prep_data["knowledge_graph"] = create_building_graph()
    new_prep_data["shares"] = None
    sec_bld = Sector("bld", new_prep_data)
    return sec_bld

In [4]:
sectors = [get_vema_prep(), get_buildings_prep()]
ns_coordinates = {
    "Region": sectors[0].coordinates["Region"],
    "material": list(set(sectors[0].coordinates["material"] + sectors[1].coordinates["material"]))
}
ns_combined = Sector("combi", {}, coordinates=ns_coordinates)
sectors.append(ns_combined)

In [5]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [6]:
REGION = prism.Dimension("Region")
MATERIAL_TYPE = prism.Dimension("material")

@prism.interface
class SumMaterials(prism.Model):
    Region: prism.Coords[REGION]
    material: prism.Coords[MATERIAL_TYPE]

    input_data: tuple[str] = ("inflow_materials", )
    output_data: tuple[str] = ("summed_inflow_materials", )



    summed_inflow_materials: prism.TimeVariable[REGION, MATERIAL_TYPE, "count"] = prism.export()

    def compute_initial_values(self, time: prism.Timeline):
        pass

    def compute_values(self, time: prism.Time, inflow_materials):
        t, dt = time.t, time.dt
        self.summed_inflow_materials[t].loc[:] = 0
        for inflow in inflow_materials:
            self.summed_inflow_materials[t].loc[{"material": inflow[t].coords["material"]}] += inflow[t].sum("Type")

In [7]:
factory = ModelFactory(
    sectors, complete_timeline
    ).add(GenericStocks, ["bld", "vhc"]
    ).add(GenericMaterials,  "vhc"
    ).add(MaterialIntensities, "bld",
    ).add(SumMaterials, "combi", input_sources={"inflow_materials": ["vhc", "bld"]}
    )
model = factory.finish()

In [8]:
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    model.simulate(simulation_timeline)

In [9]:
model.combi["summed_inflow_materials"][2020]

<xarray.DataArray (Region: 26, material: 18)> Size: 4kB
<Quantity([[1.41219760e+05 1.22718536e+08 1.26421021e+07 0.00000000e+00
  0.00000000e+00 0.00000000e+00 1.32121698e+08 0.00000000e+00
  1.77949525e+09 3.14053541e+04 8.45956745e+03 0.00000000e+00
  1.31866859e+07 0.00000000e+00 2.49731139e+01 1.90178474e+08
  0.00000000e+00 0.00000000e+00]
 [2.43765396e+06 3.29474556e+09 4.35661733e+08 0.00000000e+00
  2.67188064e+01 0.00000000e+00 4.07612934e+09 0.00000000e+00
  5.56866794e+10 1.26120988e+05 5.87994803e+04 0.00000000e+00
  3.44316587e+08 0.00000000e+00 1.76238028e+02 6.08529717e+09
  0.00000000e+00 0.00000000e+00]
 [7.44073936e+04 4.29192175e+08 1.23440039e+07 0.00000000e+00
  0.00000000e+00 0.00000000e+00 1.49164797e+08 0.00000000e+00
  2.20953720e+09 2.41481618e+04 6.57689128e+03 0.00000000e+00
  5.16500873e+07 0.00000000e+00 1.87623275e+01 3.21846480e+08
  0.00000000e+00 0.00000000e+00]
 [1.91752187e+05 3.10515752e+08 3.31483124e+07 0.00000000e+00
  0.00000000e+00 0.00000000e+00 3.88872476e+08 0.00000000e+00
  5.04646915e+09 1.28002039e+04 3.52176177e+03 0.00000000e+00
  3.92035956e+07 0.00000000e+00 1.00467599e+01 6.50026312e+08
  0.00000000e+00 0.00000000e+00]
...
 [4.37862531e+07 2.03347838e+09 5.62676339e+08 5.14556167e+07
  0.00000000e+00 2.72026240e+05 4.26353334e+08 0.00000000e+00
  7.84267257e+09 7.25211039e+04 2.49599996e+04 0.00000000e+00
  3.58619778e+08 0.00000000e+00 1.00300513e+02 1.54365429e+09
  1.09791684e+07 0.00000000e+00]
 [2.83213502e+05 1.71603105e+08 8.86361878e+06 0.00000000e+00
  0.00000000e+00 0.00000000e+00 1.05635266e+08 0.00000000e+00
  1.48115090e+09 3.05630521e+04 9.82962960e+03 0.00000000e+00
  2.18693258e+07 0.00000000e+00 1.41121574e+01 2.21690450e+08
  0.00000000e+00 0.00000000e+00]
 [7.54695471e+04 1.72956648e+09 4.12408604e+07 0.00000000e+00
  3.49003417e-01 0.00000000e+00 4.61470991e+08 0.00000000e+00
  7.18765150e+09 1.19138897e+05 3.74134862e+04 0.00000000e+00
  1.98666952e+08 0.00000000e+00 1.28646570e+02 1.07126035e+09
  0.00000000e+00 0.00000000e+00]
 [5.27642056e+06 7.74285216e+08 8.26742635e+07 6.11566374e+06
  3.35001547e+00 3.23311840e+04 2.18714752e+08 0.00000000e+00
  3.53049416e+09 3.39531728e+04 1.08650826e+04 0.00000000e+00
  1.06248565e+08 0.00000000e+00 3.83090432e+01 5.69288099e+08
  1.30490909e+06 0.00000000e+00]], 'count')>
Coordinates:
  * Region    (Region) <U2 208B '1' '2' '3' '4' '5' ... '22' '23' '24' '25' '26'
  * material  (material) <U9 648B 'Wood' 'Aluminium' 'Cu' ... 'Nd' 'Li'